# 13 — Streamlit AI Forecast Chatbot & Analytics Dashboard

This notebook launches an interactive **Streamlit web application** directly within Google Colab without needing external tunneling services like ngrok.

### Features Included:
1. **🤖 AI Agent Chatbot**: Natural language interface querying Finnish quarterly vacancy forecasts with Qwen3-4B, citing official Ministry/KEHA bulletins.
2. **📊 Figures & Analytics Dashboard**: Immediate display of primary national and regional figures, with an interactive dropdown selector for model comparisons and diagnostic plots.
3. **🎨 Custom Aesthetics**: Theme switcher (Dark / Light mode) with an **RGB animated border** glowing between light coral (`#F08080`) and light blue (`#ADD8E6`).

In [ ]:
# Step 1: Connect to Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
    print("Connected to project root:", os.getcwd())
except ImportError:
    print("Running in local environment.")

In [ ]:
# Step 2: Install Streamlit and required inference dependencies
%pip install -q streamlit chromadb sentence-transformers pyyaml pandas "transformers==5.17.0" "peft==0.20.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "safetensors==0.8.0" "huggingface-hub==1.31.0"

In [ ]:
# Step 2b: Verify or Download Base Model (Downloads once, ~2-3 mins, stored permanently in Google Drive)
import os, sys
from pathlib import Path

REPO = Path(os.environ.get("JOBAI_REPO", "/content/drive/MyDrive/JobAI")).resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from jobai.model_runtime import base_directory

print("Checking Qwen3-4B base model cache in Google Drive...")
base_path = base_directory(REPO, "Qwen/Qwen3-4B", download=True)
print(f"\u2705 Base model is cached and ready at: {base_path}")


In [ ]:
# Step 3: Launch Streamlit in the background & expose port via Colab proxy
import subprocess, sys, time, os
from urllib.request import urlopen
from pathlib import Path

REPO = Path(os.environ.get("JOBAI_REPO", Path.cwd())).resolve()
APP_PATH = REPO / "apps" / "streamlit_app.py"
if not APP_PATH.is_file():
    APP_PATH = REPO / "streamlit_app.py"
assert APP_PATH.is_file(), f"Application script not found at {APP_PATH}"

# Stop any previously running Streamlit instance
if "STREAMLIT_PROCESS" in globals() and STREAMLIT_PROCESS.poll() is None:
    print("Stopping earlier Streamlit process...")
    STREAMLIT_PROCESS.terminate()
    STREAMLIT_PROCESS.wait(timeout=10)

LOG_FILE = REPO / "reports" / "streamlit_runtime.log"
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
log_handle = open(LOG_FILE, "w", encoding="utf-8")

print("Starting Streamlit dashboard on port 8501...")
STREAMLIT_PROCESS = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(APP_PATH),
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--browser.gatherUsageStats=false"
    ],
    cwd=str(REPO),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

# Wait up to 30 seconds for health check endpoint
for attempt in range(30):
    try:
        with urlopen("http://127.0.0.1:8501/_stcore/health", timeout=1) as resp:
            if resp.read() == b"ok":
                print("Streamlit is UP and running!")
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"Streamlit failed to start. Review logs at {LOG_FILE}")

# Expose port using Colab's native tunnel
try:
    from google.colab.output import eval_js, serve_kernel_port_as_iframe
    from IPython.display import HTML, display

    proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
    print("\n" + "="*60)
    display(HTML(f'<h3>🚀 <a href="{proxy_url}" target="_blank" style="color:#2563EB; font-weight:bold;">Click here to open JobAI Dashboard in a new tab</a></h3>'))
    print("="*60)
    print("Embedding preview iframe below...")
    serve_kernel_port_as_iframe(8501, height=850)
except ImportError:
    print("Open http://localhost:8501 in your browser.")